In [1]:
# Packages, seed and path
## packages
from edge_sim_py import *
import math
import os
import random
import msgpack
import pandas as pd
import json
import numpy as np
from numpy import random

## path
import os
abspath = os.path.abspath(os.path.join('..'))
algo_name = "tetris"

In [2]:
# change the slas
def change_slas(path, mu, sigma):
    # Opening JSON file
    f = open(path)
    
    # returns JSON object as 
    # a dictionary
    data = json.load(f)
    
    # Iterating through the json
    # sla values from probabilistic distribution
    n = len(data['User'])
    sla_values = np.around(np.random.normal(mu, sigma, n), decimals=0)
    # list
    for i in range(len(data['User'])):
        print('>>> user', i, "<<<")
        print('before:', data['User'][i].get('attributes').get('delay_slas'))
        data['User'][i].get('attributes').get('delay_slas').update({str(i+1): sla_values[i]})
        print('after:', data['User'][i].get('attributes').get('delay_slas'))

    # Closing file
    f.close()

    # Serializing json
    json_object = json.dumps(data, indent=4)
    
    # Writing to sample.json
    with open(path, "w") as outfile:
        outfile.write(json_object)

# change the network delays
def change_network(path, link_delay, wireless_delay):
    # Opening JSON file
    f = open(path)
    
    # returns JSON object as 
    # a dictionary
    data = json.load(f)
    
    # Iterating through the json - networklink
    for i in range(len(data['NetworkLink'])):
        print('>>> network link', i, "<<<")
        print('before:', data['NetworkLink'][i].get('attributes').get('delay'))
        data['NetworkLink'][i].get('attributes').update({'delay': link_delay})
        print('after:', data['NetworkLink'][i].get('attributes').get('delay'))

    # Iterating through the json - wireless
    for i in range(len(data['BaseStation'])):
        print('>>> base station', i, "<<<")
        print('before:', data['BaseStation'][i].get('attributes').get('wireless_delay'))
        data['BaseStation'][i].get('attributes').update({'wireless_delay': wireless_delay})
        print('after:', data['BaseStation'][i].get('attributes').get('wireless_delay'))
    # Closing file
    f.close()

    # Serializing json
    json_object = json.dumps(data, indent=4)
    
    # Writing to sample.json
    with open(path, "w") as outfile:
        outfile.write(json_object)

In [3]:
# link_delay = 10 
# wireless_delay = 10
# scenario = '1st_low_slow'
# path = f"{abspath}/datasets/dataset_{scenario}.json"
# change_network(path, link_delay, wireless_delay)
# scenario = '1st_high_slow'
# path = f"{abspath}/datasets/dataset_{scenario}.json"
# change_network(path, link_delay, wireless_delay)

In [4]:
# Custom collect method to measure users
def user_custom_collect_method(self) -> dict: 
    # Python libraries
    import copy
    """Method that collects a set of metrics for the object.

    Returns:
        metrics (dict): Object metrics.
    """
    access_history = {}
    for app in self.applications:
        access_history[str(app.id)] = self.access_patterns[str(app.id)].history

    try: 
        delay_sla_deadlines = self.delay_slas.get(str(self.id)) - self.delays.get(str(self.id)) 
    except: 
         delay_sla_deadlines = None

    metrics = {
        "Instance ID": self.id,
        "Coordinates": self.coordinates,
        "Base Station": f"{self.base_station} ({self.base_station.coordinates})" if self.base_station else None,
        "Delays": self.delays.get(str(self.id)), #copy.deepcopy(self.delays),
        "Delay slas": self.delay_slas.get(str(self.id)),#copy.deepcopy(self.delay_slas),
        "Delay sla deadlines": delay_sla_deadlines,
        "Communication Paths": copy.deepcopy(self.communication_paths),
        "Making Requests": copy.deepcopy(self.making_requests),
        "Access History": copy.deepcopy(access_history)
    }
    return metrics

def server_custom_collect_method(self) -> dict:
        """Method that collects a set of metrics for the object.

        Returns:
            metrics (dict): Object metrics.
        """
        metrics = {
            "Instance ID": self.id,
            "Coordinates": self.coordinates,
            "Available": self.available,
            "CPU": self.cpu,
            "RAM": self.memory,
            "Disk": self.disk,
            "CPU Demand": self.cpu_demand,
            "RAM Demand": self.memory_demand,
            "Disk Demand": self.disk_demand,
            "Ongoing Migrations": self.ongoing_migrations,
            "Services": [service.id for service in self.services],
            "Registries": [registry.id for registry in self.container_registries],
            "Layers": [layer.instruction for layer in self.container_layers],
            "Images": [image.name for image in self.container_images],
            "Download Queue": [f.metadata["object"].instruction for f in self.download_queue],
            "Waiting Queue": [layer.instruction for layer in self.waiting_queue],
            "Max. Concurrent Layer Downloads": self.max_concurrent_layer_downloads,
            "Power Consumption": self.get_power_consumption(),
            "rfc": self.rfc,
            "rf_cpu": self.rf_cpu,
            "rf_memory": self.rf_memory,
        }
        return metrics

In [5]:
## My min max scaler
def my_scaler(x_unscaled, x_min=None, x_max=None):
    if x_min == None:
        x_min=x_unscaled.min()
    if x_max == None:
        x_max=x_unscaled.max()
    x_scaled = (x_unscaled - x_min)/(x_max - x_min)
    return x_scaled

In [6]:
# Sorting applications according to their sla deadline
def my_algorithm(parameters):

    print(f"[STEP {parameters['current_step']}]")

    for edge_server in EdgeServer.all():
        # reset fragmentation monitoring
        edge_server.rfc = 0
        edge_server.rf_cpu = 0
        edge_server.rf_memory = 0
    
    sla_list = list()

    for app in Application.all():
        #print('>>>>>>>>>>>>>>>>>>>>>>> 1. Scheduling' + str(app))
        #print(app.services)
        user = app.users[0]
        delay = user.delays
        #print(user.making_requests)

        if delay.get(list(delay)[0]) == None: d_aux = 0
        else: d_aux = delay.get(list(delay)[0])
        
        sla = user.delay_slas
        s_aux = sla.get(list(sla)[0])

        sla_deadline = s_aux - d_aux

        sla_list.append(sla_deadline)
    
    dt_sla = pd.DataFrame(sla_list, columns=['sla_deadline']).reset_index().rename(columns={'index': 'user'}).sort_values(by='sla_deadline')
    dt_sla['user'] = dt_sla['user'] + 1
    #print(dt_sla)

    # Iterate over every app in sorted list
    for app_id in dt_sla['user']: 
        app = Application.find_by_id(app_id)

        #print('>>>>>>>>>>>>>>>>>>>>>>> 2. Placement ' + str(app))

        # Iterating over the list of services that compose the application
        for ser in app.services:
            #print('>>>>>>>>>>>>>>>>>>>>>>> 2.1 Placement ' + str(service))

            #print(ser, "being provisioned:", ser.being_provisioned)

            if not ser.being_provisioned: #If service needs to be migrated
                
                #To sort edge servers based on their available free resources, we will use Python's "sorted" method. 
                #The server capacity is represented by three layers: CPU, memory, and disk. To determine the average resource utilization of each server, 
                #we calculate the geometric mean of these three layers. Finally, we set the "reverse" attribute of the sorted method to "True," 
                #allowing us to arrange the edge servers in descending order of their free resources. (x_unscaled - x_min)/(x_max - x_min)

                edge_servers = sorted(
                    EdgeServer.all(),
                    key=lambda s: ((s.cpu - s.cpu_demand) * (s.memory - s.memory_demand) * (s.disk - s.disk_demand)) ** (1 / 3),
                    reverse=False,
                )

                for edge_server in edge_servers:
                    # Check if the edge server has resources to host the service
                    ser.drop = True
                    if edge_server.has_capacity_to_host(service=ser) & (edge_server.available):
                        # We just need to migrate the service if it's not already in the least occupied edge server
                        ser.drop = False
                        if ser.server != edge_server:
                            
                            #print(f"Migrating {ser} From {ser.server} to {edge_server}")
                            ser.provision(target_server=edge_server)
                            # After start migrating the service we can move on to the next service
                            break
                    else:
                        if ((edge_server.cpu - ser.cpu_demand) >= 0) | ((edge_server.memory - ser.memory_demand) >= 0):
                            
                            edge_server.rfc = edge_server.rfc + 1

                            print('entrou', edge_server, edge_server.rfc)
                            
                            if ((edge_server.cpu - ser.cpu_demand) >= 0):
                                edge_server.rf_cpu = edge_server.rf_cpu + (edge_server.cpu - ser.cpu_demand)
                                print(edge_server.rf_cpu)
                            
                            if ((edge_server.memory - ser.memory_demand) >= 0):
                                edge_server.rf_memory = edge_server.rf_memory + (edge_server.memory - ser.memory_demand)
                                print(edge_server.rf_memory)
                        #print(str(edge_server) + " has no capacity to host " + str(service))

def stopping_criterion_services(model: object):
    # Defining a variable that will help us to count the number of services successfully provisioned within the infrastructure
    provisioned_services = 0
    
    # Iterating over the list of services to count the number of services provisioned within the infrastructure
    for service in Service.all():

        # Initially, services are not hosted by any server (i.e., their "server" attribute is None).
        # Once that value changes, we know that it has been successfully provisioned inside an edge server.
        if service.server != None:
            provisioned_services += 1
    
    # As EdgeSimPy will halt the simulation whenever this function returns True, its output will be a boolean expression
    # that checks if the number of provisioned services equals to the number of services spawned in our simulation
    stop = False
    if (provisioned_services == Service.count()) | (model.schedule.steps == 60):
        stop = True
    return stop

def stopping_criterion_steps(model: object):    
    # As EdgeSimPy will halt the simulation whenever this function returns True,
    # its output will be a boolean expression that checks if the current time step is 'n'
    n = 60
    return model.schedule.steps == n

In [7]:
# run the experiment many times
print(f">>>>>> [{algo_name}] <<<<<<")

## seed packages
from random import seed
#import torch
import numpy as np

## defining 10 numbers for seed_list
seed_list = [428956419, 1954324947, 1145661099, 1835732737, 794161987, 1329531353, 200496737, 633816299, 1410143363, 1282538739]
## defining scenarios
scenario_list = ['sample', '2ec_low_on', '2ec_low_off', '2ec_high_on', '2ec_high_off']
#scenario_list = ['sample']

# Simulation execution
## 10 different seed to execute 10x the experiment
for seed_value in seed_list:
    print(f"====== [SEED {seed_value}] ======")
    seed(seed_value)
    #torch.manual_seed(seed_value)
    np.random.seed(seed_value)

    ## 4 different scenarios that will be executed by 10 different seeds
    for scenario in scenario_list:
        print(f">>> [scenario {scenario}] <<<")

        if scenario == 'sample':
            mu, sigma = 40, 2.5 # mean and standard deviation
            # start the simulation
            simulator = Simulator(
                tick_duration=1,
                tick_unit="seconds",
                stopping_criterion=stopping_criterion_steps,
                resource_management_algorithm=my_algorithm,
                logs_directory=f"logs/algorithm={algo_name}/seed={seed_value}/scenario={scenario}",
            )
        else:
            mu, sigma = 15, 5 # mean and standard deviation
            # start the simulation
            simulator = Simulator(
                tick_duration=1,
                tick_unit="seconds",
                stopping_criterion=stopping_criterion_services,
                resource_management_algorithm=my_algorithm,
                logs_directory=f"logs/algorithm={algo_name}/seed={seed_value}/scenario={scenario}",
            )

        # Loading a sample dataset
        ## change the slas
        path = f"{abspath}/datasets/dataset_{scenario}.json"
        change_slas(path, mu, sigma)
        simulator.initialize(input_file=f"{abspath}/datasets/dataset_{scenario}.json")
        #simulator.initialize(input_file="https://raw.githubusercontent.com/EdgeSimPy/edgesimpy-tutorials/master/datasets/sample_dataset2.json")

        # Assigning the custom collect methods
        User.collect = user_custom_collect_method
        EdgeServer.collect = server_custom_collect_method
        
        # Executing the simulation
        simulator.run_model()

        # Saving Results -------------------------------------------------------------------------------------------------------------------------------------
        logs_edgeserver = pd.DataFrame(simulator.agent_metrics["EdgeServer"])
        logs_edgeserver.to_csv(f"results/{algo_name}_logsserver_{scenario}_{seed_value}.csv", index=False)

        logs_service = pd.DataFrame(simulator.agent_metrics["Service"])
        logs_service.to_csv(f"results/{algo_name}_logsservice_{scenario}_{seed_value}.csv", index=False)
        
        logs_user = pd.DataFrame(simulator.agent_metrics["User"])
        logs_user.to_csv(f"results/{algo_name}_logsuser_{scenario}_{seed_value}.csv", index=False)

>>>>>> [tetris] <<<<<<
====== [SEED 428956419] ======
>>> [scenario sample] <<<
>>> user 0 <<<
before: {'1': 40.0}
after: {'1': 37.0}
>>> user 1 <<<
before: {'2': 43.0}
after: {'2': 40.0}
[STEP 1]
[STEP 2]
[STEP 3]
entrou EdgeServer_3 1
4
4096
entrou EdgeServer_4 1
4
4096
entrou EdgeServer_1 1
4
12288
entrou EdgeServer_2 1
4
12288
entrou EdgeServer_4 2
10
10240
[STEP 4]
[STEP 5]
entrou EdgeServer_3 1
4
4096
entrou EdgeServer_4 1
4
4096
entrou EdgeServer_1 1
4
12288
entrou EdgeServer_2 1
4
12288
entrou EdgeServer_4 2
10
10240
[STEP 6]
[STEP 7]
entrou EdgeServer_3 1
4
4096
entrou EdgeServer_4 1
4
4096
entrou EdgeServer_1 1
4
12288
entrou EdgeServer_2 1
4
12288
entrou EdgeServer_4 2
10
10240
[STEP 8]
[STEP 9]
entrou EdgeServer_3 1
4
4096
entrou EdgeServer_4 1
4
4096
entrou EdgeServer_1 1
4
12288
entrou EdgeServer_2 1
4
12288
entrou EdgeServer_4 2
10
10240
[STEP 10]
[STEP 11]
entrou EdgeServer_3 1
4
4096
entrou EdgeServer_4 1
4
4096
entrou EdgeServer_1 1
4
12288
entrou EdgeServer_2 1
4
122